In [3]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

INPUT_DIR = "./dataset/public"
OUTPUT_PATH = "./working/submission.csv"

CLEAN_COLS = ["transient_class", "host_environment", "spectral_regime"]
DERIVED_COLS = ["variability_pattern", "distance_bin", "energy_tier", "followup_protocol", "precursor_category"]
LABEL_COLS = CLEAN_COLS + DERIVED_COLS
N_FOLDS = 5

def extract_numeric_features(df):
    """
    DOMAIN PRIORS: Extract the structured numerical attributes.
    Fixed regex to strictly ignore trailing grammatical periods.
    """
    df = df.copy()
    
    # \d+(?:\.\d+)? strictly matches numbers like "0.224" or "3" and ignores trailing punctuation
    df['redshift'] = df['narrative'].str.extract(r'z\s*=\s*(\d+(?:\.\d+)?)').astype(float)
    df['luminosity'] = df['narrative'].str.extract(r'log L\s*=\s*(\d+(?:\.\d+)?)').astype(float)
    
    # Fill missing values with a designated outlier (-999) for XGBoost
    df['redshift'] = df['redshift'].fillna(-999)
    df['luminosity'] = df['luminosity'].fillna(-999)
    
    return df

def main():
    print("Step 1: Loading Data...")
    train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
    test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))
    
    train_df = extract_numeric_features(train_df)
    test_df = extract_numeric_features(test_df)
    
    # ==========================================
    # STAGE 1: Extract Clean Labels for Test Set
    # ==========================================
    print("Step 2: Training NLP Extractors for Clean Labels...")
    for col in CLEAN_COLS:
        # TF-IDF + LogReg captures verbatim class mentions perfectly
        pipeline = make_pipeline(
            TfidfVectorizer(ngram_range=(1, 3), max_features=5000), 
            LogisticRegression(C=10, max_iter=1000, random_state=42)
        )
        pipeline.fit(train_df['narrative'], train_df[col])
        test_df[col] = pipeline.predict(test_df['narrative'])
        
        # Verify extraction accuracy via training score
        train_acc = pipeline.score(train_df['narrative'], train_df[col])
        print(f"  -> {col} Extractor Train Accuracy: {train_acc:.4f}")

    # ==========================================
    # STAGE 1.5: Fix - Safely Label Encode
    # ==========================================
    label_encoders = {}
    for col in LABEL_COLS:
        le = LabelEncoder()
        # Fit ONLY on train_df since it contains all valid classes
        le.fit(train_df[col])
        train_df[col + "_enc"] = le.transform(train_df[col])
        
        # Only transform test_df for the Clean Cols we just extracted
        if col in CLEAN_COLS:
            test_df[col + "_enc"] = le.transform(test_df[col])
            
        label_encoders[col] = le

    # ==========================================
    # STAGE 2: Train Tabular Predictor (XGBoost)
    # ==========================================
    print("\nStep 3: Training Tabular XGBoost Models for Derived Labels...")
    features = [col + "_enc" for col in CLEAN_COLS] + ["redshift", "luminosity"]
    
    # Prepare OOF (Out-of-Fold) arrays and test predictions
    test_predictions = {col: np.zeros((len(test_df), len(label_encoders[col].classes_))) for col in DERIVED_COLS}
    
    # We enforce Compositional CV to ensure robust evaluation
    pairs = train_df[['transient_class', 'host_environment']].drop_duplicates().values.tolist()
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    
    for fold, (train_idx_pairs, val_idx_pairs) in enumerate(kf.split(pairs)):
        print(f"\n--- FOLD {fold + 1} ---")
        val_pairs = [pairs[i] for i in val_idx_pairs]
        val_mask = train_df.apply(lambda row: [row['transient_class'], row['host_environment']] in val_pairs, axis=1)
        
        X_train = train_df[~val_mask][features]
        X_val = train_df[val_mask][features]
        
        for col in DERIVED_COLS:
            y_train = train_df[~val_mask][col + "_enc"]
            y_val = train_df[val_mask][col + "_enc"]
            
            # REGULARIZATION: Shallow trees (max_depth=3) prevent memorizing the 12% noise
            xgb_model = XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="mlogloss",
                early_stopping_rounds=30,
                random_state=42 + fold
            )
            
            xgb_model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
            
            # Predict and evaluate OOF
            val_preds = xgb_model.predict(X_val)
            val_f1 = f1_score(y_val, val_preds, average="macro")
            print(f"  [{col}] Fold Val Macro F1: {val_f1:.4f}")
            
            # Accumulate test predictions (probabilities)
            test_predictions[col] += xgb_model.predict_proba(test_df[features]) / N_FOLDS

    # ==========================================
    # STAGE 3: Format Submission
    # ==========================================
    print("\nStep 4: Compiling Final Submission...")
    submission_df = pd.DataFrame({'id': test_df['id']})
    
    # Add Clean Labels (from Stage 1)
    for col in CLEAN_COLS:
        submission_df[col] = test_df[col]
        
    # Add Derived Labels (from Stage 2)
    for col in DERIVED_COLS:
        final_class_indices = np.argmax(test_predictions[col], axis=1)
        submission_df[col] = label_encoders[col].inverse_transform(final_class_indices)
        
    # Ensure exact column order
    submission_df = submission_df[["id"] + LABEL_COLS]
    
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    submission_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Done! Predictions saved to {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

Step 1: Loading Data...
Step 2: Training NLP Extractors for Clean Labels...
  -> transient_class Extractor Train Accuracy: 1.0000
  -> host_environment Extractor Train Accuracy: 1.0000
  -> spectral_regime Extractor Train Accuracy: 1.0000

Step 3: Training Tabular XGBoost Models for Derived Labels...

--- FOLD 1 ---
  [variability_pattern] Fold Val Macro F1: 0.5360
  [distance_bin] Fold Val Macro F1: 0.6410
  [energy_tier] Fold Val Macro F1: 0.7714
  [followup_protocol] Fold Val Macro F1: 0.7196
  [precursor_category] Fold Val Macro F1: 0.6547

--- FOLD 2 ---
  [variability_pattern] Fold Val Macro F1: 0.5447
  [distance_bin] Fold Val Macro F1: 0.6670
  [energy_tier] Fold Val Macro F1: 0.7145
  [followup_protocol] Fold Val Macro F1: 0.6985
  [precursor_category] Fold Val Macro F1: 0.7028

--- FOLD 3 ---
  [variability_pattern] Fold Val Macro F1: 0.5131
  [distance_bin] Fold Val Macro F1: 0.6378
  [energy_tier] Fold Val Macro F1: 0.7203
  [followup_protocol] Fold Val Macro F1: 0.6990
  [